# HPML Chapter 2 — Lab: A Methodology for Performance Engineering

**Dr. Kaoutar El Maghraoui**

This notebook is the hands-on companion to Chapter 2. Section numbers map one-to-one to the chapter's subsections. By the end you will have run the full RAG case-study methodology on a real Colab T4: latency distribution (§2.6.1), stage-level profile (§2.6.2), per-kernel profiler output (§2.3.3), Nsight Systems timeline capture (§2.3.4), Roofline placement (§2.4.3), and the performance specification (§2.6.4).

### What you will produce

1. A latency distribution (p50 / p95 / p99) over 100 queries.
2. A stage-level waterfall matching Figure 2.9.
3. A per-kernel PyTorch profiler ranking matching Table 2.4.
4. A Chrome trace file you can open at https://ui.perfetto.dev.
5. An Nsight-compatible `.nvtx` annotated trace, for the lucky few with a local NVIDIA workstation. The Colab workflow produces an equivalent Perfetto trace.
6. A Roofline analysis with measured arithmetic intensity.
7. A regression-test harness (§2.5.3) that fails if decode latency regresses by more than 15%.

### Before you start

Set **Runtime → Change runtime type → Hardware accelerator: T4 GPU.** The labs will run on CPU too, but the timings will be meaningless and the Roofline placement will be wrong.

## 0  Setup

Install the stack. The `-q` flag keeps the output tidy. About 90 seconds on a cold Colab runtime.

In [ ]:
# Install quantization/accelerate libs and a small retrieval stack.
# sentence-transformers for BGE, faiss-cpu for the vector store,
# transformers for Gemma-2B, accelerate for device_map='auto'.
!pip install -q -U torch==2.4.1 torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.45.2 accelerate==0.34.2 sentence-transformers==3.1.1 faiss-cpu==1.8.0
!pip install -q datasets==3.0.1 bitsandbytes==0.44.1

In [ ]:
# Confirm the GPU is a T4 and PyTorch sees CUDA.
import torch, subprocess
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f'  HBM:      {props.total_memory / 1e9:.1f} GB')
    print(f'  SMs:      {props.multi_processor_count}')
    print(f'  Compute:  sm_{props.major}{props.minor}')
print()
print(subprocess.getoutput('nvidia-smi | head -20'))

If `Device` is anything other than Tesla T4, the absolute latency numbers in the chapter will not match yours, but the methodology and relative ranking will be identical. The Roofline parameters below auto-detect your GPU and adjust.

## 1  Build the naïve RAG baseline (§1.4, §2.6.1)

The smallest honest RAG system: BGE-small embeddings, a FAISS `IndexFlatIP` vector store over a ~2 000-passage Wikipedia subset, and Gemma-2B-IT as the generator with no quantization, batch size 1, 20 output tokens. Deliberately inefficient. This is the baseline the rest of the book chips away at.

In [ ]:
import os, time, json, gc
from dataclasses import dataclass
from typing import List, Tuple
import numpy as np
import torch
import torch.nn.functional as F

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float16 if DEVICE == 'cuda' else torch.float32
print(f'Using {DEVICE} / {DTYPE}')

In [ ]:
# --- Tiny corpus: 2000 passages from wikitext-2 ---
from datasets import load_dataset
ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
# Keep paragraphs of reasonable length
corpus = [t.strip() for t in ds['text'] if 200 < len(t) < 800]
corpus = corpus[:2000]
print(f'Corpus size: {len(corpus)} passages')

In [ ]:
# --- Embedding model: BGE-small (much smaller than BGE-M3 so Colab
#     can host both the embedder and Gemma-2B at once) ---
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer('BAAI/bge-small-en-v1.5', device=DEVICE)
print(f'Embedding dim: {embedder.get_sentence_embedding_dimension()}')

In [ ]:
# --- Build FAISS index (one-time, ~20s) ---
import faiss
embeddings = embedder.encode(
    corpus, batch_size=64, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True,
)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype('float32'))
print(f'Indexed {index.ntotal} passages of dim {dim}')

In [ ]:
# --- Generator: Gemma-2B-IT, FP16, no quantization ---
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_ID = 'google/gemma-2b-it'  # Gated; accept the license at huggingface.co first

# Uncomment if you need to authenticate:
# from huggingface_hub import login; login()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map=DEVICE,
    attn_implementation='sdpa',  # gets FlashAttention-2 on Ampere+, SDPA fallback on T4
)
model.eval()
print(f'Model params: {sum(p.numel() for p in model.parameters()) / 1e9:.2f} B')
print(f'Model dtype: {next(model.parameters()).dtype}')

### The RAG pipeline (single-request function)

This is the function we will profile from several angles. Note the explicit stage boundaries. They are what makes the stage-level profile in §2.6.2 possible.

In [ ]:
@dataclass
class RAGResult:
    query: str
    retrieved: List[str]
    answer: str
    stage_times: dict  # stage name -> milliseconds
    total_ms: float

def rag_answer(query: str, top_k: int = 5, max_new_tokens: int = 20) -> RAGResult:
    stages = {}
    t_total_start = time.perf_counter()

    # --- Stage 1: encode query ---
    t0 = time.perf_counter()
    q_emb = embedder.encode(query, normalize_embeddings=True, convert_to_numpy=True)
    torch.cuda.synchronize() if DEVICE == 'cuda' else None
    stages['encode_query'] = (time.perf_counter() - t0) * 1000

    # --- Stage 2: FAISS retrieval ---
    t0 = time.perf_counter()
    q_emb = q_emb.reshape(1, -1).astype('float32')
    scores, ids = index.search(q_emb, top_k)
    retrieved = [corpus[i] for i in ids[0]]
    stages['retrieval'] = (time.perf_counter() - t0) * 1000

    # --- Stage 3: prompt assembly + tokenization ---
    t0 = time.perf_counter()
    context = '\n\n'.join(retrieved)
    prompt = (
        f'Context:\n{context}\n\nQuestion: {query}\nAnswer:'
    )
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
    stages['tokenize_prompt'] = (time.perf_counter() - t0) * 1000

    # --- Stage 4 + 5: prefill + decode (split via CUDA events for GPU time) ---
    if DEVICE == 'cuda':
        start = torch.cuda.Event(enable_timing=True)
        mid   = torch.cuda.Event(enable_timing=True)
        end   = torch.cuda.Event(enable_timing=True)
        start.record()
        with torch.no_grad():
            # prefill = forward with use_cache, capture past_key_values
            out = model(**inputs, use_cache=True)
            past = out.past_key_values
            next_token = out.logits[:, -1:].argmax(dim=-1)
            mid.record()
            # decode loop (greedy)
            generated = [next_token]
            for _ in range(max_new_tokens - 1):
                out = model(input_ids=next_token, past_key_values=past, use_cache=True)
                past = out.past_key_values
                next_token = out.logits[:, -1:].argmax(dim=-1)
                generated.append(next_token)
            end.record()
        torch.cuda.synchronize()
        stages['prefill'] = start.elapsed_time(mid)
        stages['decode']  = mid.elapsed_time(end)
    else:
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id)
        stages['prefill_plus_decode'] = (time.perf_counter() - t0) * 1000
        generated = [out[:, inputs['input_ids'].shape[1]:]]

    # --- Stage 6: detokenize ---
    t0 = time.perf_counter()
    gen_ids = torch.cat(generated, dim=1)[0]
    answer = tokenizer.decode(gen_ids, skip_special_tokens=True)
    stages['detokenize'] = (time.perf_counter() - t0) * 1000

    total_ms = (time.perf_counter() - t_total_start) * 1000
    return RAGResult(query=query, retrieved=retrieved, answer=answer,
                     stage_times=stages, total_ms=total_ms)

# --- Smoke test ---
r = rag_answer('What is the capital of France?')
print('Answer:', r.answer[:200])
print('Total ms:', round(r.total_ms, 1))
print('Stages:', {k: round(v, 1) for k, v in r.stage_times.items()})

## 2  Lab: latency distribution over 100 queries (§2.6.1)

The distribution-discipline rule from §2.2.1: report p50/p95/p99, not just the mean. This lab runs 100 queries (10 warmup, 90 measured) and produces the table that maps to Table 2.6 of the chapter.

In [ ]:
# A small eval set of factual questions
EVAL_QUERIES = [
    'What is the capital of France?',
    'Who wrote Pride and Prejudice?',
    'What year did World War II end?',
    'What is the speed of light?',
    'Who painted the Mona Lisa?',
    'What is the largest planet in the solar system?',
    'What is the chemical symbol for gold?',
    'Who was the first president of the United States?',
    'What is the longest river in the world?',
    'What is the boiling point of water?',
]
# Repeat to get 100 queries total
EVAL_QUERIES = EVAL_QUERIES * 10
print(f'Eval set: {len(EVAL_QUERIES)} queries')

In [ ]:
# --- Warmup (discard first 10 results) ---
print('Warmup (10 iterations)...')
for q in EVAL_QUERIES[:10]:
    _ = rag_answer(q)
torch.cuda.synchronize() if DEVICE == 'cuda' else None

# --- Measurement loop ---
print('Measuring (100 iterations)...')
totals, stage_log = [], []
for q in EVAL_QUERIES:
    r = rag_answer(q)
    totals.append(r.total_ms)
    stage_log.append(r.stage_times)
totals = np.array(totals)

print()
print('Latency distribution (ms):')
print(f'  mean      {totals.mean():8.1f}')
print(f'  std       {totals.std():8.1f}')
print(f'  p50       {np.percentile(totals, 50):8.1f}')
print(f'  p95       {np.percentile(totals, 95):8.1f}')
print(f'  p99       {np.percentile(totals, 99):8.1f}')

**Interpretation.** Compare your p50 / p95 / p99 against the chapter's baseline (Table 2.6: p50 = 5710 ms, p95 = 7420 ms). Your numbers will differ. We use Gemma-2B here, not 2B + 5-passage BGE-M3 retrieval, and BGE-small is about 4× faster than BGE-M3.

What matters structurally is the p99/p50 ratio. It should land somewhere around 1.3 to 1.5 for a well-behaved baseline. A much higher ratio means something is hitchhiking on your run: GPU contention from another process, thermal throttling on a free-tier T4, or (most commonly) you forgot the warmup loop.

## 3  Lab: stage-level waterfall (§2.6.2, Fig 2.9)

Principle #3 in practice: profile end-to-end before drilling down. We aggregate the per-stage timings collected in the previous cell and visualize them as a stacked waterfall bar.

In [ ]:
import pandas as pd

df = pd.DataFrame(stage_log)
summary = df.median().round(1)
total_median = summary.sum()
summary_pct = (summary / total_median * 100).round(1)

print('Stage-level profile (median over 100 measured iterations):')
print()
print(f'{"Stage":<22} {"Median (ms)":>12} {"% of total":>12}')
print('-' * 48)
for stage, ms in summary.items():
    pct = summary_pct[stage]
    print(f'{stage:<22} {ms:>12.1f} {pct:>11.1f}%')
print('-' * 48)
print(f'{"TOTAL":<22} {total_median:>12.1f} {100.0:>11.1f}%')

# Identify the dominant stage
dominant = summary.idxmax()
print(f'\nDominant stage: {dominant} ({summary_pct[dominant]:.1f}% of total)')

In [ ]:
# --- Visualize as a stacked waterfall bar (Figure 2.9 style) ---
import matplotlib.pyplot as plt

TERRA = '#B85C2C'
TERRA_D = '#8B3F18'
TERRA_L = '#D88058'
GRAY = '#5C5C5C'
GRAY_L = '#A0A0A0'
CREAM = '#FAF6F2'

# Color by bottleneck class
stage_colors = {
    'encode_query':     TERRA_L,   # compute-bound
    'retrieval':        GRAY,      # memory-bound (DRAM)
    'tokenize_prompt':  GRAY_L,    # CPU
    'prefill':          TERRA,     # compute-bound
    'decode':           TERRA_D,   # memory-bandwidth-bound
    'prefill_plus_decode': TERRA_D,
    'detokenize':       GRAY_L,    # CPU
}

fig, ax = plt.subplots(figsize=(12, 3))
cursor = 0
for stage, ms in summary.items():
    color = stage_colors.get(stage, CREAM)
    ax.barh(0, ms, left=cursor, color=color, edgecolor='black', linewidth=0.8)
    if ms / total_median > 0.04:
        txt_color = 'white' if color in (TERRA_D, TERRA, GRAY) else 'black'
        ax.text(cursor + ms/2, 0, f'{stage}\n{ms:.0f} ms ({summary_pct[stage]:.0f}%)',
                ha='center', va='center', fontsize=9, color=txt_color,
                fontweight='bold')
    cursor += ms

ax.set_xlim(0, total_median * 1.02)
ax.set_ylim(-0.6, 0.6)
ax.set_yticks([])
ax.set_xlabel('Latency (ms)')
ax.set_title(f'Where the time goes  —  RAG naïve baseline  (total = {total_median:.0f} ms)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
plt.tight_layout()
plt.show()

**Observation.** Almost certainly your `decode` (or `prefill_plus_decode`) stage takes 60 to 80% of the bar. That is exactly the diagnosis of Table 2.7. The decode loop dominates.

**Principle #3 in action:** we have not yet looked inside any stage. The end-to-end profile alone wipes out the entire optimization search space outside the dominant stage. Now we drill in.

## 4  Lab: PyTorch profiler — per-operator breakdown (§2.3.3)

Now we look inside `decode`. The PyTorch profiler with the `schedule()` API skips warmup and captures the steady-state operator distribution. This produces the real version of Chapter 2 Table 2.4.

In [ ]:
from torch.profiler import profile, schedule, ProfilerActivity

# Prepare a representative input: tokenize one of the eval queries with retrieved context
demo_query = 'What is the capital of France?'
demo_result = rag_answer(demo_query)  # also warms everything up
demo_context = '\n\n'.join(demo_result.retrieved)
demo_prompt  = f'Context:\n{demo_context}\n\nQuestion: {demo_query}\nAnswer:'
demo_inputs  = tokenizer(demo_prompt, return_tensors='pt', truncation=True, max_length=512).to(DEVICE)
print('Prompt length (tokens):', demo_inputs['input_ids'].shape[1])

In [ ]:
# Decode-only function that runs a fixed number of decode steps against
# a pre-computed KV cache. Isolates the memory-bandwidth-bound region.
def decode_n_steps(inputs, n_steps=20):
    with torch.no_grad():
        out = model(**inputs, use_cache=True)
        past = out.past_key_values
        next_token = out.logits[:, -1:].argmax(dim=-1)
        for _ in range(n_steps - 1):
            out = model(input_ids=next_token, past_key_values=past, use_cache=True)
            past = out.past_key_values
            next_token = out.logits[:, -1:].argmax(dim=-1)
    return next_token

# Run the profiler with the standard schedule
prof_schedule = schedule(wait=1, warmup=2, active=5, repeat=1)

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    schedule=prof_schedule,
    record_shapes=True,
) as prof:
    for step in range(8):  # wait(1) + warmup(2) + active(5) = 8
        decode_n_steps(demo_inputs, n_steps=20)
        prof.step()

# Print the top operators by Self CUDA time
print(prof.key_averages(group_by_input_shape=True).table(
    sort_by='self_cuda_time_total', row_limit=15
))

**Interpretation (compare to Table 2.4).**

- The top ~5 rows should all be `aten::mm` (matrix multiply) variants with different input shapes. These are the linear layers (`down_proj`, `gate_proj`, `up_proj`, `o_proj`, `qkv_proj`), each called many times.
- One row should be `aten::scaled_dot_product_attention` or `aten::_flash_attention_forward`. Attention is typically **under 10%** of total CUDA time at decode batch 1.
- Rows for `layer_norm`, `silu`, `add` should each be under 3%.

**Verdict:** the linears dominate, exactly as the chapter claims. Attention is not the near-term optimization target. Weight streaming is.

In [ ]:
# Export the Chrome trace for Perfetto viewing.
prof.export_chrome_trace('/content/rag_decode_trace.json')
print('Trace saved to /content/rag_decode_trace.json')
print()
print('To view: open https://ui.perfetto.dev in another tab,')
print('         click "Open trace file", select the downloaded file.')
print()

# Download it in Colab
try:
    from google.colab import files
    files.download('/content/rag_decode_trace.json')
except ImportError:
    import os
    size_mb = os.path.getsize('/content/rag_decode_trace.json') / 1e6
    print(f'Trace size: {size_mb:.1f} MB')

**Reading the Chrome trace (Figure 2.3 in the chapter).** In Perfetto:

1. The top track is **Python / PyTorch operators**. You'll see nested operators: `nn.Linear.forward` enclosing `aten::linear` enclosing the CUDA kernel launch.
2. The **CUDA Runtime** track shows `cudaLaunchKernel` ticks, which are host-side kernel issuance.
3. The **CUDA HW Stream** track shows actual kernel execution.
4. Look for the **kernel launch gap** between the Runtime tick and the HW kernel start. Typically 5 to 20 µs. For healthy decode, the HW stream is densely packed with no long gaps between consecutive kernels.
5. Use Perfetto's **Flow events** view (or scroll horizontally) to see how multiple decode steps repeat their kernel pattern.

## 5  Lab: Nsight Systems — NVTX instrumentation + timeline (§2.3.4)

Nsight Systems is the tool of choice for device-level timeline analysis. Colab does not ship the `nsys` binary and cannot install it (no sudo). But the **NVTX instrumentation itself runs fine** and produces a trace that shows up labeled in Perfetto. This lab teaches the instrumentation discipline. To capture a real `.nsys-rep` file, re-run the instrumented code on any local Linux box with NVIDIA Nsight Systems installed (free download).

In [ ]:
import torch.cuda.nvtx as nvtx
from torch.profiler import profile, ProfilerActivity

def decode_with_nvtx(inputs, n_steps=5):
    """Decode loop with NVTX ranges around each step and sub-stage."""
    with torch.no_grad():
        out = model(**inputs, use_cache=True)
        past = out.past_key_values
        next_token = out.logits[:, -1:].argmax(dim=-1)
        for step in range(n_steps):
            nvtx.range_push(f'decode_step_{step}')
            out = model(input_ids=next_token, past_key_values=past, use_cache=True)
            past = out.past_key_values
            next_token = out.logits[:, -1:].argmax(dim=-1)
            nvtx.range_pop()
    return next_token

# NVTX markers show up in the PyTorch profiler's Chrome trace too
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
) as prof:
    _ = decode_with_nvtx(demo_inputs, n_steps=5)

prof.export_chrome_trace('/content/rag_decode_nvtx_trace.json')
print('NVTX-annotated trace saved to /content/rag_decode_nvtx_trace.json')
print('Open in Perfetto: decode_step_0, decode_step_1, ... appear as labeled ranges.')

### To capture a real .nsys-rep file

On a local NVIDIA Linux workstation with Nsight Systems 2024+ installed, save this notebook's code to `rag_serve.py` and run:

```bash
nsys profile \
    --trace=cuda,nvtx,osrt,cudnn,cublas \
    --sample=cpu \
    --capture-range=cudaProfilerApi \
    --output=rag_decode_trace \
    python rag_serve.py
```

Open `rag_decode_trace.nsys-rep` in the Nsight Systems GUI. You will see GPU utilization, NVTX ranges, CUDA API, HW Stream, and OS/Python tracks exactly as shown in Chapter 2 Figure 2.4.

## 6  Lab: Roofline placement of the decode stage (§2.4.3, §2.6.3)

The Roofline diagnosis: place the dominant stage on a (arithmetic intensity, achievable FLOPs/s) plot and check that it sits on the memory-bandwidth-bound diagonal. This lab computes arithmetic intensity analytically and drops the measured point onto the plot.

In [ ]:
# Hardware parameters (T4; adapt if you have something else)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'

HARDWARE = {
    'Tesla T4':           {'peak_tflops_fp16': 65.0,  'peak_bw_TB': 0.320, 'label': 'T4'},
    'Tesla V100':         {'peak_tflops_fp16': 125.0, 'peak_bw_TB': 0.900, 'label': 'V100'},
    'A100':               {'peak_tflops_fp16': 312.0, 'peak_bw_TB': 2.039, 'label': 'A100'},
    'H100':               {'peak_tflops_fp16': 989.0, 'peak_bw_TB': 3.350, 'label': 'H100'},
    'L4':                 {'peak_tflops_fp16': 121.0, 'peak_bw_TB': 0.300, 'label': 'L4'},
}
# Fuzzy match (e.g. 'Tesla T4' contains 'T4')
hw = None
for key, val in HARDWARE.items():
    if key in gpu_name or val['label'] in gpu_name:
        hw = val; break
if hw is None:
    hw = HARDWARE['Tesla T4']
    print(f'Unknown GPU {gpu_name}, using T4 parameters')
else:
    print(f'Matched {gpu_name} -> {hw["label"]}')

peak_flops = hw['peak_tflops_fp16'] * 1e12
peak_bw    = hw['peak_bw_TB'] * 1e12
ridge_ai   = peak_flops / peak_bw
print(f'Peak FLOPs: {peak_flops/1e12:.0f} TFLOP/s')
print(f'Peak BW:    {peak_bw/1e12:.2f} TB/s')
print(f'Ridge AI:   {ridge_ai:.0f} FLOP/byte')

In [ ]:
# --- Analytical AI of one decode step for Gemma-2B ---
# Approximations: ignore activation storage (small vs weights), ignore KV cache
# (small for short context), count only dense matmul FLOPs + weight reads.

n_params = sum(p.numel() for p in model.parameters())
bytes_per_param = 2  # FP16

# A forward pass over the transformer body does ~2*N_params FLOPs per token
# (one multiply-add per param per activation, with seq_len=1 at decode).
flops_per_decode = 2 * n_params
bytes_per_decode = n_params * bytes_per_param   # each weight read once
ai = flops_per_decode / bytes_per_decode

print(f'Model params:   {n_params / 1e9:.2f} B')
print(f'FLOPs/decode:   {flops_per_decode / 1e9:.1f} GFLOP')
print(f'Bytes/decode:   {bytes_per_decode / 1e9:.2f} GB')
print(f'AI:             {ai:.2f} FLOP/byte')
print()
print(f'Ridge AI:       {ridge_ai:.0f} FLOP/byte')
print(f'\nThe decode AI ({ai:.2f}) is {ridge_ai/ai:.0f}x below the ridge.')
print(f'Decode is structurally memory-bandwidth-bound on {hw["label"]}.')

In [ ]:
# --- Measure achieved throughput and place on the Roofline ---
import time

n_tokens = 20
n_trials = 5
times = []
for _ in range(3):                               # warmup
    _ = decode_n_steps(demo_inputs, n_steps=n_tokens)
torch.cuda.synchronize()
for _ in range(n_trials):
    start = torch.cuda.Event(enable_timing=True)
    end   = torch.cuda.Event(enable_timing=True)
    start.record()
    _ = decode_n_steps(demo_inputs, n_steps=n_tokens)
    end.record()
    torch.cuda.synchronize()
    times.append(start.elapsed_time(end) / 1000)  # to seconds

t_per_token = np.median(times) / n_tokens
achieved_flops = flops_per_decode / t_per_token
achieved_bw    = bytes_per_decode / t_per_token
roof_ceiling   = min(peak_flops, peak_bw * ai)

print(f'Time per decode step: {t_per_token * 1000:.1f} ms')
print(f'Achieved FLOPs/s:     {achieved_flops / 1e12:.2f} TFLOP/s')
print(f'Achieved BW:          {achieved_bw / 1e12:.2f} TB/s  ({100 * achieved_bw / peak_bw:.0f}% of peak)')
print(f'Roofline ceiling:     {roof_ceiling / 1e12:.2f} TFLOP/s')
print(f'Utilization vs ceiling: {100 * achieved_flops / roof_ceiling:.0f}%')

In [ ]:
# --- Plot the Roofline with the measured point ---
import matplotlib.pyplot as plt

TERRA = '#B85C2C'; TERRA_D = '#8B3F18'; GRAY = '#5C5C5C'; BORDER = '#D4C4B8'

ai_axis = np.logspace(-1, 3.2, 400)
mem_roof = peak_bw * ai_axis
cmp_roof = np.full_like(ai_axis, peak_flops)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(ai_axis, mem_roof / 1e12, color=TERRA, lw=2.5, label=f'{hw["label"]} BW = {peak_bw/1e12:.2f} TB/s')
ax.plot(ai_axis, cmp_roof / 1e12, color=TERRA_D, lw=2.5, label=f'{hw["label"]} FP16 peak = {peak_flops/1e12:.0f} TFLOP/s')
ax.axvline(ridge_ai, color=GRAY, lw=1.0, linestyle='--', alpha=0.6)

# Our measured point
ax.scatter([ai], [achieved_flops / 1e12], color='black', s=120, zorder=5, label='measured: RAG decode')
ax.annotate(f'  Gemma-2B decode\n  {achieved_flops/1e12:.2f} TFLOP/s at AI={ai:.2f}',
            xy=(ai, achieved_flops / 1e12), fontsize=10)

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Arithmetic intensity  (FLOP / byte)', fontsize=11)
ax.set_ylabel('Performance  (TFLOP/s)', fontsize=11)
ax.set_title(f'Roofline: {hw["label"]} — RAG decode')
ax.grid(True, which='both', color=BORDER, lw=0.4, alpha=0.6)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

**Verification of the chapter's claim.** You should see the measured point sitting **on or very close to** the memory-bandwidth diagonal, at 50 to 100% of theoretical bandwidth. This is the Roofline-based proof that:

- The decode workload is memory-bandwidth-bound on this hardware.
- The kernels are already running close to the physical limit set by the AI of the algorithm.
- No amount of kernel-level tuning can help. Only reducing bytes (quantization, KV-cache compression) or amortizing them (batching, speculative decoding) moves the point upward.

## 7  Lab: lock in the baseline with a regression test (§2.5.3)

Principle #5: every accepted optimization ships with a test that fails if the optimization is silently undone. We freeze the current baseline by saving the measured p95 latency, then build a test that fails if a future run exceeds it by more than a margin.

In [ ]:
import json

BASELINE_FILE = '/content/rag_baseline.json'
THRESHOLD_MARGIN = 1.15   # allow up to 15% regression before failing

def save_baseline(p50, p95, p99, decode_p50):
    baseline = {
        'hardware': gpu_name,
        'model': MODEL_ID,
        'torch_version': torch.__version__,
        'p50_ms': round(p50, 1),
        'p95_ms': round(p95, 1),
        'p99_ms': round(p99, 1),
        'decode_p50_ms': round(decode_p50, 1),
    }
    with open(BASELINE_FILE, 'w') as f:
        json.dump(baseline, f, indent=2)
    return baseline

def load_baseline():
    with open(BASELINE_FILE) as f:
        return json.load(f)

def regression_test(current_decode_p50_ms):
    b = load_baseline()
    threshold = b['decode_p50_ms'] * THRESHOLD_MARGIN
    ok = current_decode_p50_ms <= threshold
    status = 'PASS' if ok else 'FAIL'
    print(f'[{status}] decode p50 = {current_decode_p50_ms:.1f} ms  '
          f'(baseline {b["decode_p50_ms"]:.1f}, threshold {threshold:.1f})')
    return ok

# --- Compute decode p50 from our stage log ---
decode_times = [s.get('decode', s.get('prefill_plus_decode', 0)) for s in stage_log]
decode_p50 = float(np.percentile(decode_times, 50))

baseline = save_baseline(
    p50=np.percentile(totals, 50),
    p95=np.percentile(totals, 95),
    p99=np.percentile(totals, 99),
    decode_p50=decode_p50,
)
print('Baseline saved:')
print(json.dumps(baseline, indent=2))

In [ ]:
# --- Run the regression test against the just-saved baseline (should PASS) ---
# Recompute decode p50 on a fresh set of iterations
fresh_decode = []
for q in EVAL_QUERIES[:30]:
    r = rag_answer(q)
    fresh_decode.append(r.stage_times.get('decode', r.stage_times.get('prefill_plus_decode', 0)))
fresh_decode_p50 = float(np.percentile(fresh_decode, 50))
regression_test(fresh_decode_p50)

# --- Simulate a regression by pretending decode took 30% longer ---
simulated_regression = fresh_decode_p50 * 1.30
print()
print('Simulating a 30% regression:')
regression_test(simulated_regression)

In a real CI setup, `regression_test` would live in a `pytest` file, run on every pull request against a pinned-hardware runner, and fail the CI job fast if the threshold is exceeded. Principle #5 is now operational. The next decode regression trips a build before anyone has to remember it might.

## 8  Exercises

These tie back to the chapter's Quiz and push it into practice. All run in under 10 minutes on a Colab T4.

**Exercise 1 (§2.2.1).** Re-run Section 2 with the warmup loop commented out. How does the p99 change? How does the p50 change? Explain the difference in terms of what warmup actually does.

**Exercise 2 (§2.3.3).** Re-run the profiler in Section 4 with `with_stack=True`. Compare the operator attribution. Is any operator that was prominent without `with_stack` now split across multiple call sites? What does that tell you about where optimization effort should focus?

**Exercise 3 (§2.4.1, Amdahl).** Using your measured stage profile from Section 3, compute the maximum end-to-end speedup you could get if `decode` time dropped to zero and every other stage stayed fixed. Is that above or below the chapter's target of 4 to 5× (§2.6.4)? What does the answer say about whether decode optimization alone is enough?

**Exercise 4 (§2.4.3, Roofline).** Recompute the Roofline point with Gemma-2B loaded in INT8 instead of FP16. Use `bitsandbytes` to quantize: `AutoModelForCausalLM.from_pretrained(MODEL_ID, load_in_8bit=True, device_map=DEVICE)`. What is the new arithmetic intensity? Does the workload cross the ridge? What is the predicted speedup on a memory-bandwidth-bound decode?

**Exercise 5 (§2.6.5, trajectory).** Run the full Section 2 measurement harness against the INT8 model from Exercise 4. Measure the actual p50 speedup. Does it match the chapter's projection in Table 2.10 (baseline 5710 ms → INT8 1940 ms)? Where does your number land, and what explains the gap?

**Exercise 6 (§2.5.3).** Add a second regression test that checks p95 (not just decode p50). Combine both tests in a single function that returns True only if both pass. This is the production form of the regression harness.

## What you have just done

You have run the full Chapter 2 methodology end-to-end on real hardware:

1. Built a RAG system (§1.4).
2. Measured a latency distribution, not just a mean (§2.2.1).
3. Profiled end-to-end before drilling in (§2.1, Principle #3).
4. Used the PyTorch profiler to break the dominant stage down by operator (§2.3.3).
5. Used NVTX instrumentation to produce a trace readable in Perfetto or Nsight Systems (§2.3.4).
6. Placed the dominant stage on the Roofline and verified it sits at the memory-bandwidth ceiling (§2.4.3, §2.6.3).
7. Locked in the baseline with a regression test (§2.5.3, Principle #5).

Chapter 3 onwards takes this baseline and starts moving it toward the specification in §2.6.4. Every chapter's lab notebook will start by loading this baseline and end by running a regression test against it. The methodology is no longer a chapter argument. It is the spine of the rest of the book.